In [26]:
import ollama
import json
import re
from collections import deque

In [27]:
with open("data/grids.json", "r") as f:
    grids = json.load(f)

In [28]:
MODEL = "qwen3.5:2b"

In [29]:
SYSTEM_PROMPT = """
You are a robot path planner operating in a 5×5 grid world.

Environment
    - Coordinates are represented as [row, column].
    - Rows and columns are indexed from 0 to 4.
    - Row indices increase from top to bottom.
    - Column indices increase from left to right.

Movement rules:
    - NORTH: [row - 1, column]
    - SOUTH: [row + 1, column]
    - EAST:  [row, column + 1]
    - WEST:  [row, column - 1]
    
    Each action moves the robot exactly one cell. Only the specified coordinate changes for each action.

Constraints
    - Never enter an obstacle cell.
    - Never leave the grid boundaries.
    - Each action moves exactly one cell.
    - Return the possible next action.
    - Any valid action is acceptable.

Planning procedure:
    1. Start at the given starting position.
    2. Select an action that moves toward the goal while avoiding obstacles.
    3. Never choose an action that leaves the grid or enters an obstacle.
    4. A feedback will be provided after each action to indicate whether the action was valid or invalid.
    5. If the action was valid, plan the next action based on the robot position.
    6. If the action was invalid, plan a different action.

Output format
    - Return only a valid JSON object.
    - The move direction must be one of the following: "NORTH", "SOUTH", "EAST", or "WEST".
    - Example response: {"action": "SOUTH"}
    - Do not include explanations, Markdown fences, intermediate positions, or additional text.

"""

In [30]:
def ask_slm(messages):
    # print("Asking SLM...")
    response = ollama.chat(
        model=MODEL,
        messages=messages,
        think=False,
        options={"temperature": 0.0, "seed": 42}
    )

    return response["message"]["content"]

In [31]:
def validate_json(json_str):
    json_str = json_str.strip()
    json_str = re.sub(r"^```(?:json)?\s*", "", json_str)
    json_str = re.sub(r"\s*```$", "", json_str)
    try:
        data = json.loads(json_str)
        if isinstance(data, dict) and "action" in data:
            return True, data['action']
        else:
            return False, "INVALID_JSON"
    except json.JSONDecodeError:
        return False, "INVALID_JSON"

In [32]:
MOVES = {
    "NORTH": (-1, 0),
    "SOUTH": (1, 0),
    "EAST": (0, 1),
    "WEST": (0, -1),
}

In [33]:
def validate_move(position, action, obstacles, grid_size=5):
    action = action.upper()

    if action not in MOVES:
        return {
            "valid": False,
            "reason": "Unknown action"
        }

    row, col = position
    dr, dc = MOVES[action]
    next_position = (row + dr, col + dc)

    if not (
        0 <= next_position[0] < grid_size
        and 0 <= next_position[1] < grid_size
    ):
        return {
            "valid": False,
            "reason": "Move leaves grid boundaries"
        }

    if next_position in {tuple(p) for p in obstacles}:
        return {
            "valid": False,
            "reason": "Move enters an obstacle"
        }

    return {
        "valid": True,
        "next_position": list(next_position)
    }

In [34]:
def get_neighboring_obstacles(position, obstacles):
    row, col = position
    neighboring_obstacles = []

    for dr, dc in MOVES.values():
        neighbor = (row + dr, col + dc)
        if neighbor in {tuple(p) for p in obstacles}:
            neighboring_obstacles.append(list(neighbor))

    return neighboring_obstacles

In [35]:
def bfs(start, goal, obstacles, grid_size=5):
    queue = deque([(start, [])])
    visited = set()
    visited.add(tuple(start))

    while queue:
        current_position, path = queue.popleft()

        if current_position == goal:
            return path

        for action, (dr, dc) in MOVES.items():
            next_position = [current_position[0] + dr, current_position[1] + dc]

            if (
                0 <= next_position[0] < grid_size
                and 0 <= next_position[1] < grid_size
                and tuple(next_position) not in visited
                and next_position not in obstacles
            ):
                visited.add(tuple(next_position))
                queue.append((next_position, path + [action]))

    return None

In [ ]:
solved_grids = 0
solution_details = []

for grid_info in grids:
    grid = grid_info["grid"]

    start_position = grid_info["start"]
    current_position = start_position
    goal_position = grid_info["goal"]
    obstacles = grid_info["obstacles"]
    moves_sequence = []
    total_moves = 0
    attempted_moves = 0
    path_difference = 0
    is_solved = False
    has_invalid_move = False

    

    print(f"Grid:\n{'\n'.join(' '.join(row) for row in grid)}")

    initial_neighboring_obstacles = get_neighboring_obstacles(start_position, obstacles)

    initial_prompt = f"Starting position: {start_position}\nGoal position: {goal_position}\nNeightboring Obstacles: {initial_neighboring_obstacles}"
    


    messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": initial_prompt}
        ]


    while attempted_moves < 25:
        print(f"Attempted moves: {attempted_moves}, Total executed moves: {total_moves}, Current position: {current_position}")
        attempted_moves += 1
        system_message = messages[0]
        recent_messages = messages[1:][-4:]

        context = [
            system_message,
            *recent_messages
        ]

        response = ask_slm(context)
        print(f"SLM response: {response}")
        
        is_valid, action = validate_json(response)

        if not is_valid:
            print(f"Invalid JSON response: {response}. Retrying...")


            messages.append({
                "role": "user",
                "content": (
                    f"The action was invalid.\n"
                    f"Current position: {current_position}\n"
                    "Please provide a valid JSON object with an action."
                ),
            })

            continue

        validation_result = validate_move(current_position, action, obstacles)

        if validation_result["valid"]:
            total_moves += 1
            moves_sequence.append(action)
            # print(f"Valid move: {action}.")
            current_position = validation_result['next_position']
            is_solved = current_position == goal_position

            

            if is_solved:
                solved_grids += 1
                break

            neighboring_obstacles = get_neighboring_obstacles(current_position, obstacles)

            messages.append({
                "role": "user",
                "content": (
                    f"The action was valid.\n"
                    f"Current position: {current_position}\n"
                    f"Goal position: {goal_position}\n"
                    f"Neighboring Obstacles: {neighboring_obstacles}\n"
                ),
            })

            
        else:
            has_invalid_move = True
            messages.append({
                "role": "user",
                "content": f"Invalid move: {action}.\nCurrent position: {current_position}.\n Reason: {validation_result['reason']}. Retrying..."
            })
            print(f"Invalid move: {action}. Reason: {validation_result['reason']}.")
            break

    # print(f"Response:\n{response}\n")
    print("SLM Path Planning Results:")

    if is_solved:
        print("Goal reached!")
    elif has_invalid_move:
        print("Encountered an invalid move. Stopping further attempts.")
    else:
        print("Failed to reach the goal within 25 attempts.")

    print(f"Total attempted moves: {attempted_moves}")
    print(f"Total executed moves: {total_moves}")
    # print(f"Moves sequence: {moves_sequence}")

    print("BFS Path Planning Results:")
    bfs_path = bfs(start_position, goal_position, obstacles)
    if bfs_path is not None:
        print(f"BFS path to goal: {bfs_path}")
    else:
        print("BFS failed to find a path to the goal.")

    if bfs_path is not None and is_solved:
        print("Different between SLM and BFS paths:")
        slm_set = set(moves_sequence)
        bfs_set = set(bfs_path)
        path_difference = len(moves_sequence) - len(bfs_path)
        print(f"SLM took {path_difference} more moves than BFS.")

    solution_details.append({
        "start": start_position,
        "goal": goal_position,
        "obstacles": obstacles,
        "slm_moves": moves_sequence,
        "bfs_moves": bfs_path,
        "is_solved": is_solved,
        "has_invalid_move": has_invalid_move,
        "path_difference": path_difference
    })

print(f"Total solved grids: {solved_grids} out of {len(grids)}")

Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 0]
SLM response: {"action": "SOUTH"}
Invalid move: SOUTH. Reason: Move enters an obstacle.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 1
Total executed moves: 0
BFS Path Planning Results:
BFS path to goal: ['EAST', 'SOUTH', 'SOUTH']
Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .
Attempted moves: 0, Total executed moves: 0, Current position: [0, 4]
SLM response: {"action": "SOUTH"}
Attempted moves: 1, Total executed moves: 1, Current position: [1, 4]
SLM response: {"action": "WEST"}
Invalid move: WEST. Reason: Move enters an obstacle.
SLM Path Planning Results:
Encountered an invalid move. Stopping further attempts.
Total attempted moves: 2
Total executed moves: 1
BFS Path Planning Results:
BFS path to goal: ['SOUTH', 'SOUTH', 'WEST', 'WEST', 'SOUTH', 'WEST', 'WEST']
Grid:
. . . . .
. . B B .
. .

In [47]:
def create_solution_summary(solution_details):
    total_grids = len(solution_details)
    solved_grids = 0
    solved_path_differences = 0
    solved_path_length = 0
    for detail in solution_details:
        if detail["is_solved"]:
            solved_grids += 1
            solved_path_differences += detail["path_difference"]
            solved_path_length += len(detail["slm_moves"])
    unsolved_grids = total_grids - solved_grids

    return {
        "total_grids": total_grids,
        "solved_grids": solved_grids,
        "unsolved_grids": unsolved_grids,
        "average_solved_path_difference": solved_path_differences / solved_grids if solved_grids > 0 else 0,
        "average_solved_path_length": solved_path_length  / solved_grids if solved_grids > 0 else 0
    }

In [48]:
create_solution_summary(solution_details)

{'total_grids': 100,
 'solved_grids': 54,
 'unsolved_grids': 46,
 'average_solved_path_difference': 0.0,
 'average_solved_path_length': 2.740740740740741}